# 02 — The straight-channel representation series

**Question.** Does the shape and orientation of the mesh cells change how well a narrow river
channel is represented? A straight channel that runs diagonally across a Cartesian grid is
"staircased": water has to zig-zag between square cells, which costs conveyance, so the
water surface stands higher than it should. A5 cells are pentagons with faces in many
directions, so one might expect A5 to treat all channel orientations alike. This notebook
tests that with a 20 m wide channel cut into a valley, at several orientations, on A5 meshes and
on Cartesian grids with the same cell area, with both the full shallow-water solver (SWE) and
FloodA5's default local-inertial scheme.

It covers two registry cases (`cases/registry/channel.jl`; `docs/TEST_CASES.md` §7):

| case | tier | runs | what |
|---|---|---|---|
| `channel` | standard | 16 | cells 22 and 11 m, orientations 0° and 45°, A5 and Cartesian, SWE and local-inertial |
| `channel_series` | full | 224 | cells 44, 22, 11 and 5.6 m, orientations 0–90° every 15°, two cross-channel offsets |

The findings of the full series are written up in `docs/HYDRAULICS.md` §15.12.

**Running it.** Use the kernel *Julia FloodA5 1.12* (`notebooks/README.md`). The first run of
the `channel` case makes its meshes (about a minute) and 16 runs of 3 simulated hours (about
15 minutes with two runs of three threads at a time); afterwards the notebook reuses the runs
and takes a minute or two. The full series takes many hours and is off by default
(`RUN_FULL_SERIES` below); section 5 instead reads the metrics recorded when the series was run
for HYDRAULICS §15.12.

## 1. Setup

The notebook loads three modules from the repository, all plain Julia (no Python):

- `FA` — the analysis module (`analysis/FloodA5Analysis.jl`): loads output files and computes metrics,
  including the channel metrics (`analysis/channel.jl`);
- `FC` — the test-case registry (`cases/FloodA5Cases.jl`): every case's preparation commands,
  runs, metrics and checks;
- `FP` — static figures with CairoMakie (`visualisation/FloodA5Plots.jl`), in the project's house style.

The repository root is taken from the active project (the kernel starts Julia with
`--project=<repo>`), so the notebook works from any checkout.

In [ ]:
const REPO = dirname(Base.active_project())      # the kernel's --project: this checkout
cd(REPO)
include(joinpath(REPO, "analysis", "FloodA5Analysis.jl"));   const FA = FloodA5Analysis
include(joinpath(REPO, "cases", "FloodA5Cases.jl"));          const FC = FloodA5Cases
include(joinpath(REPO, "visualisation", "FloodA5Plots.jl"));  const FP = FloodA5Plots
using CairoMakie, Printf, Statistics
CairoMakie.activate!(type = "png", px_per_unit = 1.5)
FP.activate!()                                   # house style for the figures below

RUN_CASES       = true    # make any missing `channel` runs (about 15 min the first time)
RUN_FULL_SERIES = false   # make the 224 runs of `channel_series` (many hours; see section 5)
PARALLEL, THREADS = 2, 3  # runs at a time, threads per run

## 2. The channel and its valley

Every run uses the same analytic landscape, a 2 km × 2 km square centred on 51° N, 0° E. In a
local frame with *s* along the channel and *n* across it (metres from the centre), the bed is

$$
z(s, n) = Z_0 - S\,s - S_2 \max(s - s_B, 0) +
\begin{cases} 0 & |n'| \le W/2 \quad\text{(channel)}\\
D + S_X\,(|n'| - W/2) & |n'| > W/2 \quad\text{(floodplain)}\end{cases}
\qquad n' = n - \text{offset}
$$

- a rectangular channel *W* = 20 m wide with banks *D* = 2 m high;
- a bed slope *S* = 0.1 % along the channel, steepening by *S*₂ = 2 % beyond *s*_B = 0.3 *L* = 600 m: a
  hydraulic control that keeps the outflow boundary from influencing the reach that is measured;
- a floodplain that rises away from the channel at *S*_X = 0.5 %, so that any water that leaves the
  channel drains back to it;
- a constant inflow *Q* = 20 m³/s at *s* = −0.45 *L* = −900 m, on the channel centre line; every
  edge of the domain is an open (free outflow) boundary; Manning's *n* = 0.03 everywhere.

The channel is rotated by an angle α (degrees anticlockwise from east) and may be shifted
across the cells by an offset (0 or half a cell). With these numbers the flow is in bank: the
normal depth of a wide rectangular channel, *h*ₙ = (*Q n* / (*W* √*S*))^{3/5}, is 0.97 m, under the 2 m
banks. At a steady state, the water surface along the reach should therefore be the bed plus
*h*ₙ: η(s) = *Z*₀ − *S s* + *h*ₙ. The constants are those of `FA.CHANNEL`:

In [ ]:
C  = FA.CHANNEL
hn = FA.channel_normal_depth()
U  = C.Q / (C.W * hn)
@printf("Channel %.0f m wide, banks %.0f m, bed slope %.3f; floodplain cross slope %.3f; domain %.0f m square\n",
        C.W, C.D, C.S, C.SX, C.L)
@printf("Inflow %.0f m³/s, Manning n = %.2f  →  normal depth h_n = %.3f m, velocity %.2f m/s, Froude number %.2f\n",
        C.Q, C.NMAN, hn, U, U / sqrt(9.81 * hn))

### How the meshes are made

The meshes come from one script, `test/channel_bench/make_channel_meshes.jl`, which the case runs
as its preparation step. For each A5 resolution it

1. **builds the A5 mesh** over the square with `A5Grid.mesh_for_aoi(aoi, res)` (`mesh/A5Grid.jl`). This is
   the same route as `FloodModel.jl --meshgen`: Julia samples points over the area of interest, then
   hands off to Python, `mesh/a5_bridge.py` (`mesh_for_aoi` → `generate_mesh`), which uses **pya5**
   (the A5 library) to find every cell that overlaps the square (`polygon_to_cells` with
   "overlapping" containment, then growing the set through each cell's neighbours until no
   overlapping cell is missing), fetches each cell's centre and pentagon boundary, finds
   neighbours by shared vertices (two cells are neighbours if they share exactly two vertices) and
   computes cell areas. The result is a GeoParquet mesh with an `a5-v1` cell-id scheme;
2. **measures its cell size** as √(mean cell area): 44.4, 22.2, 11.1 and 5.6 m at resolutions 17–20;
3. **builds a Cartesian grid with the same cell area** over the same square
   (`generate_cartesian_mesh_bounds`, pure Julia), so every A5 mesh has a square-cell twin with the
   same number of unknowns per unit area;
4. for every orientation and offset, **samples the analytic bed** into each cell as the mean over 256
   Halton points inside the cell polygon — the same cell mean that `sample_dem_mean!` takes from a
   raster DEM, here without a raster;
5. writes the meshes (`<mesh>_r<res>_a<angle>_o<offset>.parquet`), the inflow hydrograph `q20.csv`, a
   `manifest.csv` and the job list `jobs.txt` (one line per run, the format of `test/run_jobs.sh`).

The core of the script (the bed and the per-mesh loop):

```julia
bed(x, y, α, off) = (s = x * cosd(α) + y * sind(α); a = abs(-x * sind(α) + y * cosd(α) - off);
                     Z0 - S * s - S2 * max(s - SB, 0.0) + (a > W / 2 ? D + SX * (a - W / 2) : 0.0))
for res in RES
    a5   = A5Grid.mesh_for_aoi(aoi, res)                                   # pya5, via mesh/a5_bridge.py
    cell = sqrt(mean(A5Grid._polygon_area_m2(c.boundary) for c in a5.cells))
    cart = generate_cartesian_mesh_bounds(lon0, lat0, lon1, lat1, cell; lat_ref = LAT0)
    for (mname, m) in (("a5", a5), ("cart", cart)), α in ANGLES, of in OFFS
        sample_bed!(m, α, of * cell)                                       # Halton cell means of bed()
        save_mesh_geoparquet(m, "$OUT/$case.parquet")
        ...                                                                # manifest and jobs.txt lines
```

Note that the channel is rotated, not the mesh: the A5 and Cartesian meshes of one resolution are
the same for every orientation, only the bed sampled into them changes. Mesh generation calls
Python through PyCall, so it runs with one Julia thread (`--threads 1`; see `CLAUDE.md`).

## 3. The `channel` case: commands

`FC.show_case` prints the case's description, the preparation command (the mesh script, with the
subset of resolutions, angles and offsets this case uses) and the exact command line of every run.
Each run is a separate `FloodModel.jl` process; `{case}` paths are expanded to the case directory,
`test/case_runs/channel/`. Every run lasts 3 h of simulated time (`--sim-duration 10800`) with output
every 15 min, Manning's *n* = 0.03, and either `--flow-model swe` (full shallow-water equations) or
`--flow-model standard` (the local-inertial default, written explicitly so that a change of
defaults cannot silently change the case).

In [ ]:
c = FC.case("channel")
FC.show_case(c)

The preparation step makes the eight meshes this case needs (A5 and Cartesian × two resolutions ×
two orientations, offset 0). It is skipped when they already exist.

In [ ]:
RUN_CASES && FC.prepare(c)
manifest = FA.read_channel_manifest(joinpath(FC.case_dir(c), "manifest.csv"))
CELL = Dict(r.res => r.cell for r in manifest)              # nominal cell size (m) per A5 resolution
ncells = Dict(split(l, ",")[1] => parse(Int, split(l, ",")[7])
              for l in readlines(joinpath(FC.case_dir(c), "manifest.csv"))[2:end])
for r in manifest
    @printf("%-17s  %-4s  res %d  cell %5.2f m  angle %2.0f°  %6d cells\n",
            r.case, r.mesh, r.res, r.cell, r.angle, ncells[r.case])
end

### The meshes and the sampled bed

A few helpers for drawing cells in local metres about the domain centre (FP's map functions draw
whole runs; these draw a mesh, or a window of it, with any per-cell values).

In [ ]:
const ORIGIN = (C.LON0, C.LAT0)                  # local x east, y north (m) about the domain centre
local_polys(mesh) = FP.project_polygons(FP.map_coords(mesh; coords = :local, origin = ORIGIN), mesh)

# Draw the cells of `mesh` coloured by `values`; `window = (x0, x1, y0, y1)` keeps the cells in it.
function cells!(ax, mesh, values; window = nothing, outline = false, kw...)
    P = local_polys(mesh)
    inwin(Q) = window === nothing ||
               (maximum(Q[1, :]) >= window[1] && minimum(Q[1, :]) <= window[2] &&
                maximum(Q[2, :]) >= window[3] && minimum(Q[2, :]) <= window[4])
    keep = [i for i in eachindex(P) if inwin(P[i])]
    poly!(ax, [Point2f.(P[i][1, :], P[i][2, :]) for i in keep]; color = values[keep],
          strokewidth = outline ? 0.5 : 0, strokecolor = (FP.INK, 0.45), kw...)
end

# The channel banks (|n'| = W/2) of a channel at `angle`, shifted across by `offset` (m)
function banks!(ax, angle; offset = 0.0, color = FP.INK, kw...)
    for side in (-1, 1)
        n = side * C.W / 2 + offset
        s = [-C.L, C.L]
        lines!(ax, s .* cosd(angle) .- n * sind(angle), s .* sind(angle) .+ n * cosd(angle);
               color, linewidth = 1, linestyle = :dash, kw...)
    end
end

meshfile(mesh, res, angle) = joinpath(FC.case_dir(c), @sprintf("%s_r%d_a%02d_o00.parquet", mesh, res, angle))
MESHES = [("a5", "A5"), ("cart", "Cartesian")]

The figure shows the bed sampled into the 22 m meshes with the channel at 45°: the whole domain
(left) and a 300 m window at its centre (right), with the cell outlines and the channel banks
(dashed). The channel is about one cell wide. On the Cartesian grid it becomes a staircase of
squares joined only at their corners in places; on A5 it becomes a chain of pentagons whose
faces lie at a few fixed angles to the channel.

In [ ]:
let res = 18, angle = 45, w = 150.0
    fig = Figure(size = (900, 820))
    crange = (0.0, 16.0)
    for (i, (m, label)) in enumerate(MESHES)
        M = FA.load_mesh_parquet(meshfile(m, res, angle))
        ax1 = Axis(fig[i, 1]; aspect = DataAspect(), title = "$label, $(round(CELL[res]; digits = 1)) m cells",
                   subtitle = "bed elevation, channel at $(angle)°", ylabel = "y (m)", xlabel = i == 2 ? "x (m)" : "")
        cells!(ax1, M, M.elevation; colormap = :batlow, colorrange = crange)
        lines!(ax1, [-w, w, w, -w, -w], [-w, -w, w, w, -w]; color = :white, linewidth = 1.5)
        limits!(ax1, -C.L / 2, C.L / 2, -C.L / 2, C.L / 2)
        ax2 = Axis(fig[i, 2]; aspect = DataAspect(), title = "centre, $(Int(2w)) m window",
                   subtitle = "cell outlines; channel banks dashed", xlabel = i == 2 ? "x (m)" : "")
        cells!(ax2, M, M.elevation; window = (-w, w, -w, w), outline = true, colormap = :batlow, colorrange = crange)
        banks!(ax2, angle; color = :white)
        limits!(ax2, -w, w, -w, w)
    end
    Colorbar(fig[1:2, 3]; colormap = :batlow, limits = crange, label = "Bed elevation (m)", width = 12)
    fig
end

**Which way do the cell faces point?** Water passes between cells through their shared faces, so the
directions of the face normals matter. The histogram counts face length by the direction of its
normal (mod 180°) over the 11 m meshes. The Cartesian grid has two directions, 0° and 90°. Over a
domain this small, A5 is a periodic tiling of pentagons and its faces fall into six directions,
which are not evenly spread (HYDRAULICS §15.12 lists 2°, 41°, 103°, 146°, 155° and 173°). The
largest gap between them is about 62°, against 90° on the Cartesian grid.

In [ ]:
# Face-normal direction (degrees from east, mod 180) of every polygon edge, weighted by edge length
function face_directions(mesh)
    dirs, lens = Float64[], Float64[]
    for P in local_polys(mesh), k in 1:size(P, 2) - 1
        dx, dy = P[1, k + 1] - P[1, k], P[2, k + 1] - P[2, k]
        len = hypot(dx, dy); len > 0 || continue
        push!(dirs, mod(atand(dy, dx) + 90, 180)); push!(lens, len)
    end
    dirs, lens
end

let res = 19, edges = 0:2:180
    fig = Figure(size = (900, 300))
    for (j, (m, label)) in enumerate(MESHES)
        d, l = face_directions(FA.load_mesh_parquet(meshfile(m, res, 0)))
        frac = [sum(l[i] for i in eachindex(d) if a <= d[i] < a + 2; init = 0.0) for a in edges[1:end-1]] ./ sum(l)
        ax = Axis(fig[1, j]; title = "$label, $(round(CELL[res]; digits = 1)) m cells", xticks = 0:30:180,
                  xlabel = "Face normal direction (° from east, mod 180)", ylabel = j == 1 ? "Share of face length" : "")
        barplot!(ax, edges[1:end-1] .+ 1, frac; color = FP.MESH_STYLE[m].color, gap = 0)
        vlines!(ax, [0, 45]; color = FP.MUTED, linestyle = :dash, linewidth = 1)   # the case's channel directions
        nb = length(frac)          # peaks: bins above their neighbours (circularly) with > 2% of the length
        peaks = [edges[i] + 1 for i in 1:nb if frac[i] > 0.02 &&
                 frac[i] >= frac[mod1(i - 1, nb)] && frac[i] > frac[mod1(i + 1, nb)]]
        println("$label face-normal directions (centres of 2° bins): ", join(Int.(peaks), "°, "), "°")
    end
    fig
end

## 4. Running the `channel` case

`FC.run_jobs` starts every run that is not already complete. A run is reused if its output file
records the same command line and finished (output format 2 records the run, its inputs and the
code version); otherwise it is made again. The runs go to `test/case_runs/channel/runs/`, each
with its log.

In [ ]:
if RUN_CASES
    FC.prepare(c)
    @time FC.run_jobs(c; parallel = PARALLEL, threads = THREADS)
end

### Metrics and checks

`FC.analyse` loads the 16 runs and computes, per run, from the last two output frames (2 h 45 min
and 3 h) over the measurement reach −600 m ≤ *s* ≤ 100 m (`analysis/channel.jl`):

- **Δη** (`deta`): the mean water-surface error in the cells that overlap the channel, against the
  normal-flow surface *Z*₀ − *S s* + *h*ₙ (m). A mesh that loses conveyance needs a higher water
  surface to pass the same discharge, so lost conveyance shows as Δη > 0;
- **spill**: the area of wet (> 5 cm) floodplain cells wholly outside the channel, per metre of
  reach (m) — 0 while the flow stays in bank;
- **unsteady**: the largest change of the water surface in a channel cell over the last 15 min (m);
- **outflow**: the water leaving the domain over the last 15 min as a fraction of the inflow, from the
  run log (1 at a steady state);
- the mass balance, from the log.

`FC.check` then applies the case's physical checks (the findings of HYDRAULICS §15.12 that this
subset can test) and compares every metric with the expected values recorded in
`cases/expected/channel.toml`.

In [ ]:
m = FC.analyse(c; io = devnull)
checks_ok = FC.check(c, m)
println(checks_ok ? "\nAll checks pass." : "\nSOME CHECKS FAIL.")

The metrics of every run. The last column is Δη from the metrics recorded when the full series was
run for HYDRAULICS §15.12 (`test/wp1_runs/chan/metrics.csv`, FloodA5 0.3.x), where that file exists:
the same 16 configurations are part of the series, so this shows whether the current code
reproduces it.

In [ ]:
# Rows of a channel metrics CSV (analyse_channel.jl's metrics.csv or the case's channel_metrics.csv)
function read_metrics(path)
    hdr = split(readline(path), ",")
    map(readlines(path)[2:end]) do l
        r = Dict(zip(hdr, split(l, ",")))
        (case = String(r["case"]), mesh = String(r["mesh"]), res = parse(Int, r["res"]),
         cell = parse(Float64, r["cell_m"]), angle = parse(Float64, r["angle"]),
         offset = parse(Float64, r["offset_m"]), solver = String(r["solver"]),
         deta = parse(Float64, r["deta"]), spill = parse(Float64, r["spill"]),
         unsteady = parse(Float64, r["unsteady"]), outflow = parse(Float64, r["outflow"]))
    end
end
REC_DIR = joinpath(REPO, "test", "wp1_runs", "chan")       # the recorded series (HYDRAULICS §15.12)
rec = isfile(joinpath(REC_DIR, "metrics.csv")) ?
      Dict("chan_$(r.case)_$(r.solver)" => r for r in read_metrics(joinpath(REC_DIR, "metrics.csv"))) : Dict()

jobs = sort(c.jobs; by = j -> (j.tags[:res], j.tags[:angle], j.tags[:mesh], j.tags[:solver] == "li"))
@printf("%-5s %-5s %-5s %-15s %8s %9s %10s %8s %10s %10s\n", "cell", "angle", "mesh", "solver",
        "Δη (m)", "spill (m)", "unsteady", "outflow", "mass err.", "recorded Δη")
for j in jobs
    d, t = m[j.name], j.tags
    r = get(rec, j.name, nothing)
    @printf("%4.1f  %3.0f°   %-5s %-15s %+8.3f %9.1f %10.4f %8.4f %10.1e %10s\n",
            CELL[t[:res]], t[:angle], t[:mesh] == "a5" ? "A5" : "Cart", t[:solver] == "li" ? "local-inertial" : "SWE",
            d["deta"], d["spill"], d["unsteady"], d["outflow"], d["mass_balance_log_rel"],
            r === nothing ? "–" : @sprintf("%+.3f", r.deta))
end
if !isempty(rec)
    dmax = maximum(abs(m[j.name]["deta"] - rec[j.name].deta) for j in jobs)
    @printf("\nLargest |Δη − recorded Δη| over the 16 runs: %.1e m\n", dmax)
end

What the table shows:

- **Every run loses conveyance** (Δη > 0), and the loss falls with resolution: 0.70–1.29 m at 22 m cells,
  where the 20 m channel is about one cell wide, and 0.07–0.71 m at 11 m, about two cells wide.
- **Cartesian depends on orientation as expected.** Along the grid (0°) it is the best of all runs at
  11 m (SWE 0.07 m, local-inertial 0.12 m); at 45°, where the channel is a staircase, it is about
  0.5 m with both solvers, and at 22 m the local-inertial run is the worst of the case (1.29 m) and the
  only one that spills onto the floodplain (32 m of wet floodplain per metre of reach).
- **A5 is not orientation-free either, and its worse direction is 0°**: at 11 m, 0.71 m (SWE) and
  0.50 m (local-inertial) at 0°, against 0.50 and 0.34 m at 45°. At 45° A5 and Cartesian are equal
  with SWE (0.50 and 0.51 m); with local-inertial A5 is better (0.34 against 0.50 m).
- **Local-inertial against SWE.** At 11 m the default scheme is as close to normal flow as SWE or
  closer, except Cartesian at 0° (0.12 against 0.07 m), which is within the check's 0.15 m margin.
- **Steadiness.** All runs except one are steady to within 1% of the inflow: the 22 m Cartesian SWE run at 45°
  is still oscillating at 3 h (outflow 0.89 of the inflow, the stage moving by 0.14 m over the
  last 15 min; see the outflow figure below). Mass is conserved to round-off in all of them.
- **Reproducibility.** The current code reproduces the recorded series' Δη to within 5 × 10⁻⁶ m, and
  all 129 metrics match the expected values (recorded with 0.4.2).

### The water at 3 h

Depth maps at the end of the runs (local-inertial, 11 m cells), over a 500 m window at the centre
of the domain, which lies inside the measurement reach. Along this reach the bed and the
normal-flow water surface are parallel, so the depth in the channel should be *h*ₙ = 0.97 m
everywhere; extra depth is the stage error Δη. Cells with less than 1 cm of water show the bed in
grey; the dashed lines are the channel banks.

In [ ]:
runpath(mesh, res, angle, solver) = joinpath(FC.case_dir(c), "runs", @sprintf("chan_%s_r%d_a%02d_o00_%s.h5", mesh, res, angle, solver))
const RUNS = Dict{String,FA.Run}()
getrun(mesh, res, angle, solver) = get!(() -> FA.load_run(runpath(mesh, res, angle, solver)), RUNS,
                                        "$(mesh)_$(res)_$(angle)_$(solver)")

function depth_maps(res, solver; w = 250.0, crange = (0.0, 2.0))
    fig = Figure(size = (820, 800))
    for (i, (m, label)) in enumerate(MESHES), (j, angle) in enumerate((0, 45))
        r = getrun(m, res, angle, solver)
        p = FP.fieldmap!(fig[i, j], r; var = :depth, coords = :local, origin = ORIGIN, colorrange = crange,
                         colorbar = false, sources = false, title = "$label, channel at $(angle)°",
                         subtitle = @sprintf("%s, %.0f m cells, Δη = %+.2f m", solver == "li" ? "local-inertial" : "SWE",
                                             CELL[res], m_of(m, res, angle, solver)["deta"]))
        banks!(p.axis, angle)
        limits!(p.axis, -w, w, -w, w)
        i == 1 && hidexdecorations!(p.axis; ticks = false, grid = false)
        j == 2 && hideydecorations!(p.axis; ticks = false, grid = false)
    end
    Colorbar(fig[1:2, 3]; colormap = cgrad(FP.SEQUENTIAL), limits = crange, label = "Depth at 3 h (m)",
             width = 12, highclip = FP.SEQUENTIAL[end])
    colgap!(fig.layout, 1, 24)
    fig
end
m_of(mesh, res, angle, solver) = m[@sprintf("chan_%s_r%d_a%02d_o00_%s", mesh, res, angle, solver)]
depth_maps(19, "li")

Two things to read from the maps. First, the cells that the channel only partly covers have a
higher mean bed (part of their area is bank), so they hold less water: the light cells along the
edges of the A5 channels and the checkerboard along the Cartesian staircase at 45°. Second, the
cells wholly inside the channel are deeper than *h*ₙ = 0.97 m by about Δη: darkest in the A5
channel at 0° (Δη = 0.50 m) and the Cartesian staircase at 45° (0.50 m), lightest on the Cartesian
grid at 0° (0.12 m). At 0° the A5 channel is a ragged chain of pentagons of different shapes, while
the Cartesian channel is a uniform band of squares.

The same at 22 m cells, where the channel is about one cell wide. The 45° Cartesian run spills: a
band of shallow water on the floodplain on both sides of the staircase.

In [ ]:
depth_maps(18, "li")

### The water surface along the channel

The stage error η − (*Z*₀ − *S s* + *h*ₙ) of every wet channel cell (dots) and its mean in 50 m
bins (lines), against the distance *s* along the channel, for both solvers. The shaded band is the
measurement reach over which Δη is averaged; the inflow is at *s* = −900 m and the bed steepens
at *s* = 600 m (the normal-flow surface does not apply beyond it).

In [ ]:
# Stage error of the wet channel cells of a run's last frame, against s
function stage_error(r, angle, cell; offset = 0.0)
    k = FA.nframes(r)
    s, n = FA.channel_coords(r.mesh; angle, offset)
    h, z = FA.depth(r, k), r.mesh.elevation
    ch = [i for i in eachindex(s) if abs(n[i]) <= C.W / 2 + cell / 2 && h[i] > 0.01 && s[i] < 0.3C.L]
    s[ch], [h[i] + z[i] - (C.Z0 - C.S * s[i] + hn) for i in ch]
end
function binned(s, e; width = 50.0)
    b = floor.(s ./ width)
    ub = sort(unique(b))
    (ub .+ 0.5) .* width, [mean(e[b .== u]) for u in ub]
end

let fig = Figure(size = (980, 640))
    for (i, res) in enumerate((18, 19)), (j, angle) in enumerate((0, 45))
        ax = Axis(fig[i, j]; title = @sprintf("%.0f m cells, channel at %d°", CELL[res], angle),
                  xlabel = i == 2 ? "Distance along the channel s (m)" : "",
                  ylabel = j == 1 ? "Stage error η − η_normal (m)" : "")
        vspan!(ax, [-0.3C.L], [0.05C.L]; color = (FP.MUTED, 0.12))
        hlines!(ax, [0.0]; color = FP.MUTED, linestyle = :dash, linewidth = 1)
        for (mesh, label) in MESHES, solver in ("li", "swe")
            s, e = stage_error(getrun(mesh, res, angle, solver), angle, CELL[res])
            col = FP.MESH_STYLE[mesh].color
            solver == "li" && scatter!(ax, s, e; color = (col, 0.25), markersize = 3)
            bs, be = binned(s, e)
            lines!(ax, bs, be; color = col, linewidth = 2, linestyle = solver == "li" ? :solid : :dash)
        end
        ylims!(ax, -0.6, 2.0); xlims!(ax, -1000, 650)
    end
    Legend(fig[3, 1:2], [LineElement(color = FP.MESH_STYLE["a5"].color, linewidth = 2),
                         LineElement(color = FP.MESH_STYLE["cart"].color, linewidth = 2),
                         LineElement(color = FP.INK2, linewidth = 2),
                         LineElement(color = FP.INK2, linewidth = 2, linestyle = :dash)],
           ["A5", "Cartesian", "local-inertial (dots: its cells)", "SWE"]; orientation = :horizontal)
    fig
end

Along the measurement reach the error is close to constant, so Δη (its mean) is a fair summary.
It is a property of the channel's representation along its whole length, not of the inflow or the
outflow. Near the inflow the profiles change, and in most runs the channel is wet up to the upstream
edge of the domain (*s* = −1000 m). Towards the steepening at *s* = 600 m the water accelerates and
the error falls, as it should, since the normal-flow surface no longer applies there. On A5 the
binned error ripples along the channel with the pattern of the pentagons; on the Cartesian grid it
is smooth.

### Stage error against cell size and orientation

Δη of the 16 runs: the two cell sizes on the horizontal axis (finer to the right), one line per
mesh and orientation.

In [ ]:
let fig = Figure(size = (900, 380))
    for (j, (solver, title)) in enumerate((("swe", "Full shallow-water (SWE)"), ("li", "Local-inertial (default scheme)")))
        ax = Axis(fig[1, j]; title, xlabel = "Cell size (m)", ylabel = j == 1 ? "Stage error Δη (m)" : "",
                  xticks = ([CELL[18], CELL[19]], ["22", "11"]), xreversed = true)
        hlines!(ax, [0.0]; color = FP.MUTED, linestyle = :dash, linewidth = 1)
        for (mesh, label) in MESHES, angle in (0, 45)
            cs = [CELL[18], CELL[19]]
            d = [m_of(mesh, res, angle, solver)["deta"] for res in (18, 19)]
            st = FP.MESH_STYLE[mesh]
            lines!(ax, cs, d; color = st.color, linewidth = 2, linestyle = angle == 0 ? :solid : :dash)
            scatter!(ax, cs, d; color = st.color, marker = st.marker, markersize = 10, strokecolor = FP.SURF, strokewidth = 1.5)
        end
        xlims!(ax, 25, 9); ylims!(ax, -0.1, 1.4)
    end
    Legend(fig[1, 3], [[LineElement(color = FP.MESH_STYLE[mesh].color, linewidth = 2, linestyle = angle == 0 ? :solid : :dash),
                        MarkerElement(color = FP.MESH_STYLE[mesh].color, marker = FP.MESH_STYLE[mesh].marker, markersize = 10)]
                       for (mesh, _) in MESHES for angle in (0, 45)],
           ["$label, $(angle)°" for (_, label) in MESHES for angle in (0, 45)])
    fig
end

Halving the cell size cuts Δη by 0.3–0.8 m in every configuration (the case's check asks for
at least 0.2 m). The ranking changes with resolution: at 22 m all A5 runs and the Cartesian 45° runs
are at 1.0–1.3 m, and only Cartesian at 0° is clearly better (0.70–0.74 m). At 11 m Cartesian 0° has nearly
converged, while A5 0° is now the worst run with SWE.

### Approach to a steady state

The metrics assume the runs have settled. The outflow across the domain edges, as a fraction of the
inflow, from each run's log (the volume removed at the boundaries, differenced over 10 min):

In [ ]:
# Outflow / inflow over the trailing `window` seconds, from the log's step lines
function outflow_series(log; window = 600.0)
    e = FA.parse_log_volumes(log)
    t, v = [x.t for x in e], [x.vol_removed for x in e]
    tt, q = Float64[], Float64[]
    for i in eachindex(t)
        j = findfirst(>=(t[i] - window), t)
        t[i] - t[j] >= 0.5window || continue
        push!(tt, t[i]); push!(q, (v[i] - v[j]) / (C.Q * (t[i] - t[j])))
    end
    tt, q
end

let fig = Figure(size = (980, 600))
    for (i, res) in enumerate((18, 19)), (j, angle) in enumerate((0, 45))
        ax = Axis(fig[i, j]; title = @sprintf("%.0f m cells, channel at %d°", CELL[res], angle),
                  xlabel = i == 2 ? "Time (h)" : "", ylabel = j == 1 ? "Outflow / inflow" : "",
                  xticks = 0:0.5:3)
        hlines!(ax, [1.0]; color = FP.MUTED, linestyle = :dash, linewidth = 1)
        for (mesh, label) in MESHES, solver in ("li", "swe")
            t, q = outflow_series(replace(runpath(mesh, res, angle, solver), ".h5" => ".log"))
            lines!(ax, t ./ 3600, q; color = FP.MESH_STYLE[mesh].color, linewidth = 2,
                   linestyle = solver == "li" ? :solid : :dash)
        end
        ylims!(ax, 0, 1.2); xlims!(ax, 0, 3)
    end
    Legend(fig[3, 1:2], [LineElement(color = FP.MESH_STYLE["a5"].color, linewidth = 2),
                         LineElement(color = FP.MESH_STYLE["cart"].color, linewidth = 2),
                         LineElement(color = FP.INK2, linewidth = 2),
                         LineElement(color = FP.INK2, linewidth = 2, linestyle = :dash)],
           ["A5", "Cartesian", "local-inertial", "SWE"]; orientation = :horizontal)
    fig
end

By 3 h every run except one passes the inflow out of the domain (outflow/inflow within 1%, see the
table). The exception is Cartesian SWE at 45° with 22 m cells, which oscillates from about 1.5 h on.
The time the front takes to reach the downstream edge (the main rise) depends on the mesh. That
time is mostly the channel and valley filling up, so a mesh with a higher stage (or spill) takes longer: at 0° the A5
runs arrive last, at 45° with 22 m cells the spilling Cartesian local-inertial run.

**Water leaving upstream.** Many curves have a plateau *before* the main rise. Water leaves the
domain while the front is still travelling down the valley (at about 1 m/s it needs half an hour or more to
reach the downstream edge). It can only be leaving through the **upstream edge**, 100 m behind the
inflow point. Every edge is an open boundary, and when the stage at the inflow is high, water flows
back up the channel and out. The plateau measures this upstream loss while the reach fills:

In [ ]:
# The plateau of an outflow curve before the front reaches the downstream edge (before it first
# passes 0.8): the median of the points that are > 2% of the inflow and change by < 0.2 per hour
function early_outflow(t, q)
    i80 = findfirst(>=(0.8), q)
    i80 === nothing && return NaN
    pts = [q[i] for i in 2:i80-1 if q[i] > 0.02 && abs(q[i] - q[i-1]) / (t[i] - t[i-1]) * 3600 < 0.2]
    isempty(pts) ? 0.0 : median(pts)
end
@printf("%-5s %-5s %-5s %-15s %16s %8s\n", "cell", "angle", "mesh", "solver", "early outflow/Q", "Δη (m)")
for j in jobs
    t = j.tags
    @printf("%4.1f  %3.0f°   %-5s %-15s %16.2f %+8.3f\n", CELL[t[:res]], t[:angle], t[:mesh] == "a5" ? "A5" : "Cart",
            t[:solver] == "li" ? "local-inertial" : "SWE", early_outflow(outflow_series(FC.job_log(c, j))...), m[j.name]["deta"])
end

The upstream loss is largest exactly where the stage error is largest: A5 at 0° loses 35–69% of
the inflow upstream while the reach fills, while Cartesian local-inertial at 0° loses none. The
output files do not separate the outflow by edge, so the share lost upstream *at the steady state*
is not measured, and the case's `outflow` metric (all edges together) does not show it. If a
similar share keeps leaving upstream at 3 h, the reach carries less than *Q* = 20 m³/s. Δη is
measured against the normal-flow surface for the full 20 m³/s, so it would then *understate* the
conveyance loss of those runs, and by different amounts on different meshes. (A rough check from the
output cell velocities, reliable only where the channel follows the grid, gives 20.0 m³/s along the
reach in the Cartesian local-inertial run at 0° and about 18 m³/s in the SWE run.)

**Treat this as an open caveat on the benchmark design.** A closed upstream boundary, or an inflow further
from it, would remove the effect; recording outflow per boundary segment would measure it. It does
not change the qualitative findings below, but it is a reason to read small differences in Δη
between meshes with care.

## 5. The full series (`channel_series`)

The `channel` case is a 16-run sample of the full series, which adds:

- two more resolutions: 44 m cells (the channel half a cell wide) and 5.6 m cells (nearly four cells
  wide), to see the whole convergence;
- seven orientations, 0–90° every 15°, to see how the error depends on direction — a 0°/45°
  pair cannot separate a mesh that is worst at 45° from one with its own preferred directions;
- two cross-channel offsets (0 and half a cell), so that the result does not hang on where the
  channel happens to fall relative to the cell centres; Δη per orientation is the mean of the two.

That is 4 × 7 × 2 × 2 meshes × 2 solvers = 224 runs. The res-20 (5.6 m, 130 000 cells) SWE runs
take about half an hour each on a CPU; the recorded series ran them on a GPU (`--backend gpu`). The
case's description and its first commands:

In [ ]:
cs = FC.case("channel_series")
let io = IOBuffer()
    FC.show_case(cs; io)
    lines = split(String(take!(io)), "\n")
    i = findfirst(l -> startswith(l, "run "), lines)
    foreach(println, lines[1:i + 3])
    println("… ", count(l -> startswith(l, "run "), lines), " runs in all; the checks:")
    foreach(ch -> println("  • ", ch.name), cs.checks)
end

**Where the numbers below come from.** If `RUN_FULL_SERIES = true`, the next cell runs the case
(reusing any completed runs in `test/case_runs/channel_series/`) and its analysis writes
`channel_metrics.csv`, which the rest of this section reads. Otherwise the section reads the
metrics recorded when the series was run for HYDRAULICS §15.12: `test/wp1_runs/chan/metrics.csv`
(written by `test/channel_bench/analyse_channel.jl` from those runs, FloodA5 0.3.x output format 1;
the 16 runs it shares with the `channel` case agree with the current code — see the table in
section 4). The recorded series also has `limd` runs (local-inertial with `--momentum-damping 0.05`);
they are left out here, as in the case. If neither file exists, this section only prints how to
make it.

In [ ]:
RUN_FULL_SERIES && FC.run_case(cs; parallel = PARALLEL, threads = THREADS)
case_csv = joinpath(FC.case_dir(cs), "channel_metrics.csv")
rec_csv  = joinpath(REC_DIR, "metrics.csv")
SERIES_SRC = isfile(case_csv) ? case_csv : isfile(rec_csv) ? rec_csv : nothing
if SERIES_SRC === nothing
    println("No series metrics: run `FLOODA5_NO_VIS=1 julia --project=. cases/run.jl run channel_series -P 2 -T 4`",
            " (or set RUN_FULL_SERIES = true above).")
    series = []
else
    series = filter(r -> r.solver in ("swe", "li"), read_metrics(SERIES_SRC))
    println("Series metrics from ", relpath(SERIES_SRC, REPO), ": ", length(series), " runs (",
            join(sort(unique(r.solver for r in series)), ", "), ")")
end

### The table of HYDRAULICS §15.12

Δη per orientation (each the mean over the two offsets), its root mean square over the seven
orientations and its spread (maximum − minimum), per solver, cell size and mesh. HYDRAULICS §15.12
prints the 11 and 5.6 m rows; all four resolutions are here.

In [ ]:
function series_table(rows)
    angles = sort(unique(r.angle for r in rows))
    @printf("%-15s %5s %-5s", "solver", "cell", "mesh")
    foreach(a -> @printf(" %6s", "$(round(Int, a))°"), angles)
    @printf(" %6s %6s\n", "rms", "spread")
    for s in ("swe", "li"), res in sort(unique(r.res for r in rows)), mesh in ("a5", "cart")
        sel = [r for r in rows if r.solver == s && r.res == res && r.mesh == mesh]
        isempty(sel) && continue
        pa = [mean(r.deta for r in sel if r.angle == a) for a in angles]
        @printf("%-15s %5.1f %-5s", s == "li" ? "local-inertial" : "SWE", sel[1].cell, mesh == "a5" ? "A5" : "Cart")
        foreach(x -> @printf(" %+6.2f", x), pa)
        @printf(" %6.2f %6.2f\n", sqrt(mean(abs2, pa)), maximum(pa) - minimum(pa))
    end
end
isempty(series) || series_table(series)

### Stage error against resolution

As `test/channel_bench/plot_channel.jl` draws it: per mesh and solver, the median over the 14
orientation × offset runs at each cell size (line), and their range (band); below, the width of
floodplain that is wet per metre of reach.

In [ ]:
function plot_vs_resolution(rows)
    fig = Figure(size = (980, 640))
    for (c_, (sv, stitle)) in enumerate((("swe", "Full shallow-water (SWE)"), ("li", "Local-inertial (default scheme)"))),
        (r_, (metric, ylab)) in enumerate(((:deta, "Stage error Δη (m)"), (:spill, "Floodplain spill width (m)")))
        ax = Axis(fig[r_, c_]; xscale = log2, xlabel = r_ == 2 ? "Cell size (m, √mean area)" : "",
                  ylabel = c_ == 1 ? ylab : "", title = r_ == 1 ? stitle : "",
                  xticks = ([5.5, 11.1, 22.2, 44.4], ["5.6", "11", "22", "44"]), xreversed = true)
        hlines!(ax, [0.0]; color = FP.MUTED, linestyle = :dash, linewidth = 1)
        for (mesh, label) in MESHES
            st = FP.MESH_STYLE[mesh]
            rs = [r for r in rows if r.mesh == mesh && r.solver == sv]
            isempty(rs) && continue
            cells = sort(unique(r.cell for r in rs))
            vals = [filter(isfinite, [getfield(r, metric) for r in rs if r.cell == cz]) for cz in cells]
            med = median.(vals)
            band!(ax, cells, minimum.(vals), maximum.(vals); color = (st.color, 0.18))
            lines!(ax, cells, med; color = st.color, linewidth = 2)
            scatter!(ax, cells, med; color = st.color, marker = st.marker, markersize = 9, strokecolor = FP.SURF, strokewidth = 1.5)
        end
    end
    Legend(fig[3, 1:2], [[PolyElement(color = (FP.MESH_STYLE[m].color, 0.18)), LineElement(color = FP.MESH_STYLE[m].color, linewidth = 2),
                          MarkerElement(color = FP.MESH_STYLE[m].color, marker = FP.MESH_STYLE[m].marker, markersize = 9)]
                         for (m, _) in MESHES], ["$l (median; band = range)" for (_, l) in MESHES]; orientation = :horizontal)
    fig
end
isempty(series) || plot_vs_resolution(series)

### Stage error against orientation

The figure behind HYDRAULICS §15.12's conclusion: Δη against the channel's orientation (mean over
the two offsets), one panel per cell size, A5 and Cartesian, for each solver.

In [ ]:
function plot_vs_angle(rows)
    cells = sort(unique(r.cell for r in rows); rev = true)          # coarse to fine
    fig = Figure(size = (240 * length(cells) + 60, 650))
    for (row, (sv, stitle)) in enumerate((("swe", "Full shallow-water (SWE)"), ("li", "Local-inertial (default scheme)")))
        Label(fig[2row - 1, 1:length(cells)], stitle; halign = :left, font = :bold, color = FP.INK)
        for (k, cz) in enumerate(cells)
            ax = Axis(fig[2row, k]; title = (cz < 10 ? @sprintf("%.1f", cz) : @sprintf("%.0f", cz)) * " m cells", xticks = 0:30:90,
                      xlabel = row == 2 ? "Channel orientation (°)" : "", ylabel = k == 1 ? "Stage error Δη (m)" : "")
            hlines!(ax, [0.0]; color = FP.MUTED, linestyle = :dash, linewidth = 1)
            for (mesh, label) in MESHES
                st = FP.MESH_STYLE[mesh]
                rs = [r for r in rows if r.solver == sv && r.mesh == mesh && r.cell == cz]
                isempty(rs) && continue
                angs = sort(unique(r.angle for r in rs))
                md_ = [mean(r.deta for r in rs if r.angle == a) for a in angs]
                lines!(ax, angs, md_; color = st.color, linewidth = 2)
                scatter!(ax, angs, md_; color = st.color, marker = st.marker, markersize = 9, strokecolor = FP.SURF, strokewidth = 1.5)
            end
            ylims!(ax, -0.4, 1.5)
        end
    end
    Legend(fig[5, 1:length(cells)], [[LineElement(color = FP.MESH_STYLE[m].color, linewidth = 2),
                                      MarkerElement(color = FP.MESH_STYLE[m].color, marker = FP.MESH_STYLE[m].marker, markersize = 9)]
                                     for (m, _) in MESHES], [l for (_, l) in MESHES]; orientation = :horizontal)
    fig
end
isempty(series) || plot_vs_angle(series)

What the full series adds to the `channel` case (numbers from the table above):

- **Convergence.** For both meshes and solvers, the rms of Δη over the orientations falls from about
  1.1 m at 44 m cells to 0.84–0.94 m at 22 m, 0.29–0.49 m at 11 m and 0.06–0.15 m at 5.6 m.
- **Cartesian** is symmetric about 45° and worst near it. With SWE at 11 m it goes from 0.17–0.18 m
  along the axes to 0.53 m at 45°.
- **A5 has preferred directions of its own.** With SWE at 11 m it is worst at 0–15° (0.71 m) and best
  at 75° (0.07 m). Its spread over orientations (0.64 m) is larger than Cartesian's (0.36 m), and at
  5.6 m it is 0.34 m against 0.09 m. The face-direction histogram in section 2 shows why A5 has
  preferred directions at all. It does not explain *which* ones: A5 does worst along 0–15°, where
  it has faces at about 3° and 173°. HYDRAULICS §15.12 leaves the mechanism open.
- **Local-inertial against SWE.** The default scheme is as close to normal flow as SWE or closer
  wherever the channel is resolved. At 5.6 m its rms is 0.06 m on A5 (SWE 0.15 m) and 0.09 m on
  Cartesian (SWE 0.09 m).
- **Spill.** Only at coarse resolution and mostly with local-inertial: up to 62 m of wet floodplain
  per metre of reach on Cartesian at 44 m (45°), 14 m on A5; SWE at most 6 m (A5, 44 m); nothing from
  11 m on.

HYDRAULICS §15.12 concludes that for a straight channel 0.5–4 cells wide A5 gives no systematic
advantage over a Cartesian grid of the same cell area: it trades Cartesian's 45° weakness for
preferred directions of its own. That conclusion holds for this one location (51° N, 0° E); A5's
face directions vary between the faces of its dodecahedron, so the pattern will differ elsewhere.

### The series' checks

The `channel_series` case's checks, applied to these metrics: mass conservation (from each run's
log, where the logs are next to the metrics), steadiness, convergence of the rms error, the
local-inertial default against SWE, and spill. (The check that the job arguments match the mesh
script's `jobs.txt` only applies to runs made by the case, and is skipped for the recorded
series.) The checks are provisional: they were derived from the recorded series.

In [ ]:
if !isempty(series)
    logdir = SERIES_SRC == case_csv ? joinpath(FC.case_dir(cs), "runs") : REC_DIR
    sm = Dict{String,Dict{String,Float64}}()
    for r in series
        name = "chan_$(r.case)_$(r.solver)"
        mb = FC.log_mass_balance(joinpath(logdir, name * ".log"))     # NaN if there is no log
        sm[name] = Dict("deta" => r.deta, "spill" => r.spill, "unsteady" => r.unsteady,
                        "outflow" => r.outflow, "mass_balance_log_rel" => isnan(mb.rel) ? 0.0 : mb.rel)
    end
    sm["series"] = FC.channel_series_table(series)                                    # the rms/spread table
    for ch in cs.checks
        startswith(ch.name, "job arguments") && continue
        pass = ch.test(sm)
        println(pass ? "PASS" : "FAIL", "  ", ch.name, "\n        ", ch.detail(sm))
    end
end

## 6. Summary

- A straight 20 m channel loses conveyance on both A5 and Cartesian meshes until it is several cells
  wide. The stage error is about 1 m at one cell across and 0.06–0.15 m (rms over orientations) at
  four cells across.
- Cartesian grids are worst at 45°. A5 is **not** orientation-free: at this location it is worst
  for channels running along 0–15° and best along 60–90°, with a spread over orientations as large
  as Cartesian's or larger. Neither mesh is better overall.
- The local-inertial default represents the in-bank channel as well as full SWE, or better, once the
  channel is resolved. At coarse resolution it spills onto the floodplain more than SWE,
  especially on Cartesian grids at 45°.
- The `channel` case (16 runs, about 15 min) reproduces this subset of the recorded series to
  5 × 10⁻⁶ m and passes its checks.
- Caveat: while the reach fills, part of the inflow leaves through the upstream edge (up to about
  70% for A5 at 0°). The share lost at the steady state is not measured, and Δη may understate the
  conveyance loss where it is large (section 4).

**Pointers.**
- Case definitions and checks: `cases/registry/channel.jl`; command line: `cases/run.jl show|run|check channel`.
- The series' design, table and discussion: `docs/HYDRAULICS.md` §15.12; how to run it by hand: `docs/TEST_CASES.md` §7.
- Mesh script: `test/channel_bench/make_channel_meshes.jl`; A5 mesh generation: `mesh/A5Grid.jl` (`mesh_for_aoi`) and `mesh/a5_bridge.py`.
- Metrics: `analysis/channel.jl`; series figures as files: `test/channel_bench/plot_channel.jl`.
- Previous notebook: `01_planar.ipynb` (planar slope and the SGS embankment); next: `03_carlisle.ipynb` (a real catchment against the full shallow-water reference).